# cr-train Colab Quickstart

Colab 환경에서 `GitHub 설치 -> 학습 -> 평가`를 한 번에 따라가는 노트북입니다.

| 항목 | 값 |
|------|----|
| 데이터셋 | [`Hermanni/sen12mscr`](https://huggingface.co/datasets/Hermanni/sen12mscr) |
| 권장 런타임 | Colab GPU |
| 기본 설정 | `official` split, 짧은 학습/검증/테스트 budget |

## 1. 환경 설치

[`uv`](https://docs.astral.sh/uv/)로 가상환경을 만들어 cr-train과 의존성을 격리 설치합니다. Colab 기본 패키지와의 버전 충돌을 방지합니다.

In [ ]:
!pip install -q uv
!uv venv /root/.venv
!uv pip install -p /root/.venv git+https://github.com/smturtle2/cr-train.git

import sys
sys.path.insert(0, "/root/.venv/lib/python3.12/site-packages")

import cr_train

print(f"cr_train import ready: {cr_train.__file__}")

## 2. Hugging Face 토큰 설정

토큰 없이도 동작하지만, rate limit이 더 빨리 걸릴 수 있습니다.  
Colab Secrets에 `HF_TOKEN`을 등록하거나, 아래 입력란에 직접 입력하세요.

In [ ]:
# @title 2. Hugging Face 토큰 설정
HF_TOKEN_INPUT = ""  # @param {type:"string"}

import os

token = os.environ.get("HF_TOKEN", "") or HF_TOKEN_INPUT.strip()

if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN configured.")
else:
    print("HF_TOKEN not found -- public access works, but rate limits may be tighter.")

## 3. Import 및 device 확인

In [ ]:
import torch
from torch import nn

from cr_train import MAE, Trainer, TrainerConfig, build_sen12mscr_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## 4. DataLoader 생성

기본 split은 `official`이고, train 경로는 항상 `reshard() -> shuffle(...)` 순서를 지킵니다.

In [ ]:
train_loader, val_loader, test_loader = build_sen12mscr_loaders(
    batch_size=8,
    seed=7,
    split="official",
    shuffle_buffer_size=16,
)

print("loaders ready")

## 5. 배치 확인

기본 전처리 결과를 확인합니다:

| 채널 | 전처리 | 범위 |
|------|--------|------|
| `cloudy` / `target` | `clip(0, 10000) / 10000.0` | [0, 1] |
| `sar` | `clip(-25, 0)` -> `(x + 25) / 25` | [0, 1] |

In [ ]:
batch = next(iter(train_loader))

for key in ("sar", "cloudy"):
    t = batch["inputs"][key]
    print(f"{key:6s}  shape={tuple(t.shape)}  dtype={t.dtype}  range=[{t.min():.3f}, {t.max():.3f}]")

t = batch["target"]
print(f"target  shape={tuple(t.shape)}  dtype={t.dtype}  range=[{t.min():.3f}, {t.max():.3f}]")
print(f"metadata keys: {list(batch['metadata'].keys())}")

## 6. 모델 정의

데모용 3-layer CNN입니다. `forward(sar, cloudy)` 시그니처만 맞추면 어떤 모델이든 사용 가능합니다.

In [ ]:
class TinyCloudRemovalNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(15, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 13, kernel_size=1),
        )

    def forward(self, sar: torch.Tensor, cloudy: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([sar, cloudy], dim=1))

## 7. Trainer 구성

In [ ]:
model = TinyCloudRemovalNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=nn.MSELoss(),
    metrics=[MAE()],
    config=TrainerConfig(
        max_epochs=3,
        train_max_batches=50,
        val_max_batches=10,
        test_max_batches=5,
        checkpoint_dir="artifacts/colab-notebook",
    ),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)

## 8. 학습 실행

In [ ]:
for history in trainer.step():
    print(f"train: {history['train']}")
    print(f"val:   {history['val']}")

## 9. 테스트 평가

In [ ]:
test_metrics = trainer.test()
print(f"test: {test_metrics}")